# Week 2 · Lab 03
## Pandas Selection & Filtering

> **AI Engineering Academy** · Gamut Technology Services

Every data pipeline you build will spend most of its time **selecting rows and columns**:
"adults in core markets who spend in the top 10%," "orders from this region in this date
range." Getting selection *correct and reliable* is foundational — a subtly wrong mask
silently corrupts everything downstream. This lab drills the core tools: **boolean masks**,
`.loc`/`.iloc`, `DataFrame.query`, and reusable filter functions.

> **pandas 3.0 note.** This lab targets **pandas 3.0**, where **Copy-on-Write (CoW)** is
> always on. That *changes* the classic "chained assignment" story for the better — it's now
> predictable. Part B covers the modern behavior (older tutorials you find online will be out
> of date here).

### Learning objectives
1. Build correct boolean masks with `&`, `|`, `~` — and know exactly why **parentheses** are required.
2. Select with `df.loc[row_mask, col_list]` (labels) vs. `df.iloc` (positions).
3. Explain and avoid **chained assignment** under pandas 3.0 Copy-on-Write.
4. Use `DataFrame.query` / `eval` for readable filters, and know when to prefer masks.
5. Build **reusable filter functions** and compose them.

### Time budget — ~100 min
| Segment | Time |
|---|---|
| Setup | 5 min |
| **A.** Boolean masks & parentheses | 20 min |
| **B.** `loc`/`iloc` & chained assignment (CoW) | 25 min |
| **C.** Chained masks & reusable filters | 20 min |
| **D.** `query` & `eval` | 15 min |
| **E.** Bonus: filter a partitioned Parquet | 12 min |
| Wrap-up | 3 min |


In [ ]:
%pip install -r requirements.txt

In [ ]:
# --- Setup: build a small synthetic dataset (self-contained, seeded) --------
# Cordwell Home & Hardware online-store customers. Synthetic and clearly
# fictional; seeded so everyone gets identical data. No files or internet needed.
import warnings
import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__)

rng = np.random.default_rng(42)
n = 1_000
customers = pd.DataFrame({
    "customer_id": np.arange(n),
    "age": rng.integers(16, 80, size=n),
    # deliberately messy: three spellings of the same country
    "country": rng.choice(["US", "U.S.A.", "USA", "SG", "DE", "BR", "IN"],
                          size=n, p=[.25, .05, .10, .15, .15, .15, .15]),
    "sessions": rng.poisson(3, size=n),
    "avg_session_sec": rng.normal(300, 60, size=n).clip(30, 1500).round(1),
    "spend_usd": np.round(rng.lognormal(mean=3.0, sigma=0.7, size=n), 2),
})

def check(label, predicate):
    """Soft self-check: prints PASS/FAIL, never raises."""
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("rows:", len(customers), "| columns:", list(customers.columns))
print("note: in pandas 3.0 the 'country' column dtype is", customers["country"].dtype,
      "(not 'object')")
customers.head()

## Part A — Boolean masks & parentheses

A **boolean mask** is a Series of `True`/`False` the same length as the frame; `df.loc[mask]`
keeps the `True` rows. Combine masks with the **element-wise** operators `&` (and), `|` (or),
`~` (not) — **never** Python's `and`/`or`/`not` (those work on single values, not Series).


### A0 — Why parentheses are mandatory  *(run and read)*
The element-wise operators `&`/`|` bind **tighter** than the comparisons `>=`, `==`, etc. So
without parentheses, `age >= 18 & sessions >= 5` misparses as `age >= (18 & sessions) >= 5`.


In [ ]:
# Without parentheses -> ValueError (it tried a chained comparison on Series)
try:
    broken = customers["age"] >= 18 & customers["sessions"] >= 5
except ValueError as exc:
    print("No parens -> ValueError:", str(exc)[:60], "...")
    print("  It parsed as:  age >= (18 & sessions) >= 5")

# With parentheses -> correct
ok = (customers["age"] >= 18) & (customers["sessions"] >= 5)
print("With parens -> works. Matching rows:", int(ok.sum()))

# And using Python 'and' on Series is always an error:
try:
    _ = (customers["age"] >= 18) and (customers["sessions"] >= 5)
except ValueError as exc:
    print("Python 'and' on Series -> ValueError:", str(exc)[:55], "...")

### A1 — Build and combine masks
Create three masks and combine them, then select a subset of columns with `.loc`:
- `adults`: `age >= 18`
- `heavy_users`: `sessions >= 5`
- `us_like`: `country` is one of `"US"`, `"U.S.A."`, `"USA"`

Combine all three with `&` (mind the parentheses) into `mask`, then build
`view = customers.loc[mask, ["customer_id", "age", "country", "sessions", "avg_session_sec"]]`.


💡 **Hint.** `us_like = customers["country"].isin(["US", "U.S.A.", "USA"])`. Each comparison
that you `&` together needs its own parentheses.


In [ ]:
adults = None        # TODO: customers["age"] >= 18
heavy_users = None   # TODO: customers["sessions"] >= 5
us_like = None       # TODO: customers["country"].isin([...])

mask = None          # TODO: combine the three with &
view = None          # TODO: customers.loc[mask, [ ...columns... ]]
print(None if view is None else view.shape)

In [ ]:
check("A1: mask is a boolean Series the length of the frame",
      lambda: mask.dtype == bool and len(mask) == len(customers))
check("A1: every selected row satisfies all three conditions",
      lambda: bool(((view["age"] >= 18) & (view["sessions"] >= 5)
                    & view["country"].isin(["US", "U.S.A.", "USA"])).all()))
check("A1: view has exactly the requested columns",
      lambda: list(view.columns) == ["customer_id", "age", "country", "sessions", "avg_session_sec"])
check("A1: row count matches the mask", lambda: len(view) == int(mask.sum()))

### A2 — Negation and OR
Select customers who are **low engagement** (`sessions <= 1`) **or** **not** US-like. Build
`non_us` with `~us_like`, combine with `|`, and select `["customer_id", "country", "sessions"]`.


💡 **Hint.** `~` negates a boolean Series: `non_us = ~us_like`. Combine with `|`
(parentheses!): `low_engagement | non_us`.


In [ ]:
low_engagement = None   # TODO: sessions <= 1
non_us = None           # TODO: ~us_like
subset = None           # TODO: customers.loc[low_engagement | non_us, [...]]
print(None if subset is None else len(subset))

In [ ]:
check("A2: every row is low-engagement OR non-US (the OR held)",
      lambda: bool(((subset["sessions"] <= 1)
                    | ~subset["country"].isin(["US", "U.S.A.", "USA"])).all()))
check("A2: it's a union, so it's at least as big as either piece alone",
      lambda: len(subset) >= int((customers["sessions"] <= 1).sum()))

## Part B — `loc` vs `iloc`, and chained assignment under Copy-on-Write

`.loc` selects by **label** (column names, index labels); `.iloc` by **integer position**.
For assignment, always go through `.loc` — Part B2 shows exactly why in pandas 3.0.


### B1 — Label vs positional selection, and a safe `.loc` assignment
1. Select the first 10 rows' `customer_id` and `age` two ways: by **label** with
   `customers.loc[0:9, ["customer_id", "age"]]` and by **position** with
   `customers.iloc[0:10, [0, 1]]`.
2. Create an `is_adult` column with a single `.loc` assignment.

> ⚠️ Note the **off-by-one**: `.loc[0:9]` is *inclusive* of 9 (labels), while `.iloc[0:10]`
> is *exclusive* of 10 (positions) — both give 10 rows here only because the index is `0..n-1`.


💡 **Hint.** `customers.loc[customers["age"] >= 18, "is_adult"] = True` then set the rest to
`False` with the negated mask — or just `customers["is_adult"] = customers["age"] >= 18`.


In [ ]:
by_label = None   # TODO: customers.loc[0:9, ["customer_id", "age"]]
by_pos = None     # TODO: customers.iloc[0:10, [0, 1]]

# create is_adult via .loc (no chained assignment)
# TODO: customers.loc[customers["age"] >= 18, "is_adult"] = True
# TODO: customers.loc[customers["age"] < 18, "is_adult"] = False
print(None if by_label is None else (by_label.shape, by_pos.shape))

In [ ]:
check("B1: label and positional selections return the same 10 rows",
      lambda: by_label.reset_index(drop=True).equals(by_pos.reset_index(drop=True)))
check("B1: is_adult column was created", lambda: "is_adult" in customers.columns)
check("B1: is_adult is correct for every row",
      lambda: bool((customers["is_adult"] == (customers["age"] >= 18)).all()))

### B2 — Chained assignment under Copy-on-Write *(the modern story)*

**Chained assignment** means indexing **twice** before assigning:
`df[mask]["col"] = value`. Run the guided cell to see what pandas 3.0 does with it.


In [ ]:
# GUIDED: watch what chained assignment does in pandas 3.0.
demo = customers.copy()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    demo[demo["sessions"] > 5]["vip"] = True     # <-- chained: index twice, then assign
warning_name = caught[0].category.__name__ if caught else "none"
print("pandas warned with:", warning_name)
print("did 'vip' get created on the original frame? ->", "vip" in demo.columns)

See it? In **pandas 3.0** chained assignment raises a **`ChainedAssignmentError`** warning and
**never** modifies the original — `demo[mask]` produces a temporary **copy** (Copy-on-Write),
and you assigned into that throwaway copy. This is *more predictable* than old pandas, where
you'd sometimes get a `SettingWithCopyWarning` and sometimes silently corrupt data depending
on whether a view or copy came back.

**Your task:** reliably add a `vip` column (`True` where `sessions > 5`, else `False`) to
`customers` using a single `.loc` assignment.


💡 **Hint.** One indexing step: `customers.loc[customers["sessions"] > 5, "vip"] = True`, then
set the rest `False` — or initialize `customers["vip"] = customers["sessions"] > 5`.


In [ ]:
# TODO: create a reliable 'vip' column on `customers` using .loc (not chained assignment)
print("vip" in customers.columns and int(customers.get("vip", pd.Series(dtype=bool)).sum()))

In [ ]:
check("B2: vip column exists on the original frame", lambda: "vip" in customers.columns)
check("B2: vip is True exactly where sessions > 5",
      lambda: bool((customers["vip"] == (customers["sessions"] > 5)).all()))
check("B2: vip count matches the condition",
      lambda: int(customers["vip"].sum()) == int((customers["sessions"] > 5).sum()))

## Part C — Chained masks & reusable filters

Real filtering logic is reused and composed. Wrap conditions in small functions that each
take a DataFrame and return a mask; then `&`/`|` them together.


### C1 — Compose filter functions
Implement three predicate functions (each returns a boolean mask), then combine them to select
adults, in the **top 15% of spend**, in **core markets** (`US`, `SG`, `DE`):
- `f_is_adult(df)` → `age >= 18`
- `f_high_value(df, p=90)` → `spend_usd >= df["spend_usd"].quantile(p/100)`
- `f_core_markets(df)` → `country` in `["US", "SG", "DE"]`


💡 **Hint.** `f_high_value(customers, 85)` uses the 85th percentile:
`df["spend_usd"] >= df["spend_usd"].quantile(0.85)`. Combine with `&`.


In [ ]:
def f_is_adult(df):
    ...        # TODO: return df["age"] >= 18

def f_high_value(df, p=90):
    ...        # TODO: threshold at the p-th percentile of spend_usd; return the mask

def f_core_markets(df):
    ...        # TODO: return df["country"].isin(["US", "SG", "DE"])

mask = None    # TODO: f_is_adult(customers) & f_high_value(customers, 85) & f_core_markets(customers)
hv_core = None # TODO: customers.loc[mask, ["customer_id", "age", "country", "spend_usd"]]
print(None if hv_core is None else len(hv_core))

In [ ]:
_thr85 = customers["spend_usd"].quantile(0.85)
check("C1: all selected are adults", lambda: bool((hv_core["age"] >= 18).all()))
check("C1: all selected are high-value (>= 85th pct spend)",
      lambda: bool((hv_core["spend_usd"] >= _thr85).all()))
check("C1: all selected are in core markets",
      lambda: bool(hv_core["country"].isin(["US", "SG", "DE"]).all()))

### C2 — Normalize before matching
`str.contains` is literal: `"U.S.A."` does **not** contain the substring `"US"` (the dots break
it up), so naive matching silently misses rows. **Normalize first**: strip dots and upper-case,
then match. Build `norm` and use it to select all US customers (all three spellings).


💡 **Hint.** `norm = customers["country"].str.replace(".", "", regex=False).str.upper()` turns
`US`/`U.S.A.`/`USA` all into `"USA"`. Then `customers.loc[norm.isin(["US", "USA"])]`. Compare against the
naive `customers["country"].str.contains("US", regex=False)` to see what it misses.


In [ ]:
naive = None   # TODO: customers["country"].str.contains("US", regex=False)
norm = None    # TODO: strip "." (regex=False) and .str.upper()
us_all = None  # TODO: customers.loc[norm.isin(["US", "USA"])]
print("naive matches:", None if naive is None else int(naive.sum()),
      "| normalized US customers:", None if us_all is None else len(us_all))

In [ ]:
_us_variants = customers["country"].isin(["US", "U.S.A.", "USA"])
check("C2: normalized match captures ALL three spellings",
      lambda: len(us_all) == int(_us_variants.sum()))
check("C2: naive contains('US') misses the dotted 'U.S.A.' spelling",
      lambda: int(naive.sum()) < int(_us_variants.sum()))
check("C2: the gap equals exactly the 'U.S.A.' rows",
      lambda: int(_us_variants.sum()) - int(naive.sum()) == int((customers["country"] == "U.S.A.").sum()))

## Part D — `query` & `eval`

`DataFrame.query` filters using a **string expression** where column names are bare variables —
often more readable for multi-column boolean logic. Use `@name` to reference a Python variable.


### D1 — `query` with a Python variable
Reproduce A1's filter with `query`: adults, `sessions >= @min_sess`, and country in the US
spellings. Confirm it returns the same rows as the equivalent mask.


💡 **Hint.** `customers.query('(age >= 18) & (sessions >= @min_sess) & country in ["US","U.S.A.","USA"]')`.
Inside `query`, `in [...]` works directly and `@min_sess` pulls the Python variable.


In [ ]:
min_sess = 5
q1 = None        # TODO: customers.query('... & (sessions >= @min_sess) & country in [...]')
# equivalent mask, for comparison:
mask_equiv = (customers["age"] >= 18) & (customers["sessions"] >= min_sess) & \
             customers["country"].isin(["US", "U.S.A.", "USA"])
print("query rows:", None if q1 is None else len(q1), "| mask rows:", int(mask_equiv.sum()))

In [ ]:
check("D1: query returns the same row count as the equivalent mask",
      lambda: len(q1) == int(mask_equiv.sum()))
check("D1: query returns the same rows (same customer_ids)",
      lambda: set(q1["customer_id"]) == set(customers.loc[mask_equiv, "customer_id"]))

### D2 — When to choose `query` vs masks *(read)*
- Prefer **masks** when you need IDE autocomplete/refactoring, static typing, or arbitrary
  Python (calling your own functions, complex expressions).
- Prefer **`query`** for **readability** of multi-column boolean logic, especially in
  notebooks and presentations. Its downsides: dynamic column names are awkward, typos hide in
  the string, and Python callables aren't available inside it.


### D3 — `eval` for a computed column, then `query` on it
Use `eval` to add an `engagement = sessions * avg_session_sec` column, then select the top-10%
most-engaged customers with `query`.


💡 **Hint.** `customers = customers.eval("engagement = sessions * avg_session_sec")`. Then
`customers.query("engagement >= @customers.engagement.quantile(0.9)")`.


In [ ]:
customers = None  # TODO: customers.eval("engagement = sessions * avg_session_sec")
crit = None       # TODO: customers.query("engagement >= @customers.engagement.quantile(0.9)")
print(None if crit is None else len(crit))

In [ ]:
_p90 = customers["engagement"].quantile(0.9)
check("D3: engagement column computed correctly",
      lambda: bool(np.allclose(customers["engagement"],
                               customers["sessions"] * customers["avg_session_sec"])))
check("D3: every selected row is at/above the 90th percentile",
      lambda: bool((crit["engagement"] >= _p90).all()))
check("D3: selection is roughly the top 10%",
      lambda: 0.08 <= len(crit) / len(customers) <= 0.12)

## Part E — Bonus: filter a partitioned Parquet

Selection skills carry straight over to files. Here we generate a small **partitioned
Parquet** dataset (Cordwell orders, one file per region) and filter it — proving that a
**mask** and an equivalent **`query`** agree.


In [ ]:
# GUIDED: generate a small partitioned Parquet dataset (self-contained).
from pathlib import Path
import pyarrow as pa, pyarrow.parquet as pq

rng2 = np.random.default_rng(7)
m = 2_000
orders = pd.DataFrame({
    "order_id": np.arange(m),
    "store_region": rng2.choice(["Southeast", "Northeast", "Midwest", "West", "Southwest"], size=m),
    "order_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng2.integers(0, 540, m), unit="D"),
    "order_amount": np.round(rng2.lognormal(4.0, 0.6, m), 2),
})
part_dir = Path("artifacts/orders")
part_dir.mkdir(parents=True, exist_ok=True)
for region, g in orders.groupby("store_region"):
    pq.write_table(pa.Table.from_pandas(g, preserve_index=False),
                   part_dir / f"store_region={region}.parquet")
print("wrote partitions:", sorted(p.name for p in part_dir.glob("*.parquet")))

### E1 — Load the partitions and filter with a mask
Load every `store_region=*.parquet` file into one DataFrame, then select orders from
`["Southeast", "West"]`, dated in 2025, with `order_amount` in the **top 10%** — using a
boolean mask.


💡 **Hint.** `pd.concat([pd.read_parquet(f) for f in sorted(part_dir.glob("*.parquet"))],
ignore_index=True)`. Dates are already `datetime64`, so `orders_all["order_date"].between("2025-01-01","2025-12-31")`.


In [ ]:
orders_all = None   # TODO: concat all partition files
m_region = None     # TODO: store_region in ["Southeast", "West"]
m_date = None       # TODO: order_date between 2025-01-01 and 2025-12-31
m_amount = None     # TODO: order_amount >= its 90th percentile
subset_mask = None  # TODO: orders_all.loc[m_region & m_date & m_amount]
print(None if subset_mask is None else len(subset_mask))

In [ ]:
check("E1: loaded all 2,000 orders across partitions", lambda: len(orders_all) == 2_000)
check("E1: mask subset respects the region filter",
      lambda: bool(subset_mask["store_region"].isin(["Southeast", "West"]).all()))
check("E1: mask subset respects the amount filter",
      lambda: bool((subset_mask["order_amount"] >= orders_all["order_amount"].quantile(0.90)).all()))

### E2 — Same filter with `query`, and prove they agree
Express the **same** filter with `query` and confirm it returns the identical row count.


💡 **Hint.** Precompute the threshold: `thr = orders_all["order_amount"].quantile(0.90)`. Then
`orders_all.query('store_region in ["Southeast","West"] and "2025-01-01" <= order_date <= "2025-12-31" and order_amount >= @thr')`.


In [ ]:
thr = orders_all["order_amount"].quantile(0.90)
subset_query = None   # TODO: same filter via orders_all.query(...)
print("mask:", None if subset_mask is None else len(subset_mask),
      "| query:", None if subset_query is None else len(subset_query))

In [ ]:
check("E2: query and mask return the same row count",
      lambda: len(subset_query) == len(subset_mask))
check("E2: query and mask select the same orders",
      lambda: set(subset_query["order_id"]) == set(subset_mask["order_id"]))

## Wrap-up

You drilled the selection toolkit end to end: masks (`&`/`|`/`~` with parentheses), `.loc`
vs `.iloc`, the **pandas-3.0 chained-assignment** rules, composable filter functions,
`query`/`eval`, and filtering a partitioned Parquet. Answer in a markdown cell:

1. Show two equivalent filters — one with masks, one with `query`.
2. In pandas 3.0, what does chained assignment (`df[mask]["col"] = v`) do, and why is
   `df.loc[mask, "col"] = v` the reliable choice?
3. When is `query`/`eval` a **poor** fit (hint: dynamic column names, calling your own functions)?
4. Why must you normalize a messy categorical **before** substring matching?

### Stretch goals
- Wrap the E1 filter into a reusable `def filter_orders(df, regions, start, end, pct)` returning the mask.
- Add a `spend_tier` column (`"low"/"mid"/"high"`) with `pd.cut` on `spend_usd`, then filter by tier.
- Benchmark `query` vs the equivalent mask on a 1M-row frame with `%timeit`.


In [ ]:
print("Lab complete. customers shape:", customers.shape,
      "| columns:", list(customers.columns))